# Eksperyment: SliceAware BestOf5 v1 — głębokie wejście w Grand Slamy

## Cel
Poprzedni wariant `SliceAware` dodał ogólne cechy kontekstowe dla 3 słabych slice'ów. Ten wariant skupia się **wyłącznie** na meczach Best of 5 — najsłabszym slice'ie modelu — i wprowadza cechy specyficzne dla dystansu pięciosetowego:

## Co nowego (37 cech)
1. **Endurance score** — zagregowany wskaźnik wytrzymałości: forma Bo5 + długie mecze + average minutes
2. **Long match form** — winrate w meczach trwających >150 minut (proxy dla wytrzymałości)
3. **Best of 5 avg minutes** — średni czas trwania Bo5 gracza
4. **Best of 5 serve score / return / stability** — jakość serwisu i returnu w Bo5
5. **Best of 5 vs top30 form** — winrate w Bo5 przeciw zawodnikom z top 30
6. **Pressure serve score** — serwis w sytuacjach Bo5 (proxy presji)
7. **Tournament level strength** — `G > M > F > 500 > 250` mapowane numerycznie

## Dane dodatkowe
Potrzebujemy kolumny `minutes` z 2024.csv — jej nie ma w baseline. Ładujemy ją osobno i merge'ujemy po `match_id`.

In [ ]:
import bisect
import contextlib
import io
import os
import runpy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

HISTORY_FILES = [f"../data/sample_data/{y}.csv" for y in range(2018, 2024)]
EXTRA_CONTEXT_COLUMNS = ["minutes"]
TOURNEY_LEVEL_STRENGTH = {
    "G": 1.00, "M": 0.92, "F": 0.88, "A": 0.84,
    "500": 0.78, "250": 0.68, "D": 0.58, "O": 0.50,
}

## 1. Reuse baseline + załaduj kolumnę `minutes`

`tennis_model.py` nie używa `minutes` (nie wchodzi do `cols_base`). Ładujemy ją osobnym readerem CSV, splituje tak samo jak baseline (60/20/20 chronologicznie) i merge'uje po `match_id`.

In [ ]:
BASE_SCRIPT = Path("tennis_model.py").resolve()
captured = io.StringIO()
with contextlib.redirect_stdout(captured):
    namespace = runpy.run_path(str(BASE_SCRIPT))

base_cols = list(namespace["cols_base"])
context_base_cols = list(dict.fromkeys(base_cols + EXTRA_CONTEXT_COLUMNS))
symmetrize_data = namespace["symmetrize_data"]
baseline_search = namespace["search"]
baseline_val_acc = float(namespace["val_acc"])
baseline_test_acc = float(namespace["test_acc"])
baseline_match_accuracy = float(namespace["match_accuracy"])

def load_context_frame(csv_path):
    df = pd.read_csv(csv_path)
    df["tourney_date"] = pd.to_datetime(df["tourney_date"], format="%Y%m%d")
    df = df.sort_values(["tourney_date", "match_num"]).reset_index(drop=True)
    return df[context_base_cols].dropna(subset=base_cols).reset_index(drop=True)

# Splitt 2024 identycznie jak baseline
df_2024 = load_context_frame("../data/sample_data/2024.csv")
train_len = len(namespace["df_train_raw"])
val_len = len(namespace["df_val_raw"])
test_len = len(namespace["df_test_raw"])
assert len(df_2024) == train_len + val_len + test_len, "Mismatch w dlugosciach"

train_ctx = df_2024.iloc[:train_len].reset_index(drop=True)
val_ctx = df_2024.iloc[train_len:train_len + val_len].reset_index(drop=True)
test_ctx = df_2024.iloc[train_len + val_len:].reset_index(drop=True)
for frame in (train_ctx, val_ctx, test_ctx):
    frame["match_id"] = range(len(frame))

history_ctx = pd.concat([load_context_frame(p) for p in HISTORY_FILES], ignore_index=True)

def attach_context_columns(raw_split, ctx_split):
    return raw_split.merge(ctx_split[["match_id"] + EXTRA_CONTEXT_COLUMNS], on="match_id", how="left", validate="one_to_one")

df_train_raw = attach_context_columns(namespace["df_train_raw"].copy(), train_ctx)
df_val_raw = attach_context_columns(namespace["df_val_raw"].copy(), val_ctx)
df_test_raw = attach_context_columns(namespace["df_test_raw"].copy(), test_ctx)

print(f"Historia: {len(history_ctx)} meczow z 'minutes'")
print(f"Train/Val/Test: {train_len}/{val_len}/{test_len}")
print(f"Baseline match acc: {baseline_match_accuracy:.4f}")

## 2. PlayerHistoryIndex + module-level context

Ten wariant ma ~25 wywołań filter_player_history na mecz. Stosujemy:
- **PlayerHistoryIndex** — pre-built mapa gracz→indeksy (O(log K) lookup zamiast O(N))
- **module-level state** (`_HISTORY_INDEX`, `_HISTORY_CUTOFF`) — żeby `get_player_history` automatycznie używał indexu bez przepływania `index, cutoff` przez 25 wywołań funkcji

Po pętli `set_history_context(None, None)` czyści state.

In [ ]:
class PlayerHistoryIndex:
    __slots__ = ("_full_sequence", "_player_to_indices")
    def __init__(self, full_sequence):
        self._full_sequence = full_sequence
        winners = full_sequence["winner_name"].to_numpy()
        losers = full_sequence["loser_name"].to_numpy()
        idx = np.arange(len(full_sequence))
        combined = pd.concat([pd.Series(idx, index=winners), pd.Series(idx, index=losers)])
        self._player_to_indices = {p: np.sort(g.to_numpy()) for p, g in combined.groupby(level=0)}
    def past_for(self, player, exclusive_end):
        indices = self._player_to_indices.get(player)
        if indices is None:
            return self._full_sequence.iloc[0:0]
        cut = np.searchsorted(indices, exclusive_end, side="left")
        return self._full_sequence.iloc[indices[:cut]] if cut > 0 else self._full_sequence.iloc[0:0]

_HISTORY_INDEX = None
_HISTORY_CUTOFF = None

def set_history_context(index, cutoff):
    global _HISTORY_INDEX, _HISTORY_CUTOFF
    _HISTORY_INDEX = index
    _HISTORY_CUTOFF = cutoff

def get_player_history(player_name, history):
    if _HISTORY_INDEX is not None and _HISTORY_CUTOFF is not None:
        return _HISTORY_INDEX.past_for(player_name, _HISTORY_CUTOFF)
    return history[(history["winner_name"] == player_name) | (history["loser_name"] == player_name)]

## 3. Funkcje kontekstowe z filtrami

`filter_player_history` aplikuje serie filtrów (best_of, surface, opponent_rank_max, minutes_min) na pre-przefiltrowanej historii gracza. Z indexem ten DataFrame ma ~50-300 wierszy zamiast 20 000.

In [ ]:
def filter_player_history(player_name, history, *, best_of=None, surface=None, opponent_rank_max=None, minutes_min=None):
    ph = get_player_history(player_name, history)
    if best_of is not None:
        ph = ph[ph["best_of"] == best_of]
    if surface is not None:
        ph = ph[ph["surface"] == surface]
    if opponent_rank_max is not None:
        mask = (
            ((ph["winner_name"] == player_name) & (ph["loser_rank"] <= opponent_rank_max)) |
            ((ph["loser_name"] == player_name) & (ph["winner_rank"] <= opponent_rank_max))
        )
        ph = ph[mask]
    if minutes_min is not None:
        mm = pd.to_numeric(ph["minutes"], errors="coerce")
        ph = ph[mm >= minutes_min]
    return ph

def context_form(name, hist, *, best_of=None, surface=None, opponent_rank_max=None, minutes_min=None, window=12, min_matches=3, fallback=0.5):
    ph = filter_player_history(name, hist, best_of=best_of, surface=surface, opponent_rank_max=opponent_rank_max, minutes_min=minutes_min).tail(window)
    if len(ph) < min_matches:
        return fallback
    return float((ph["winner_name"] == name).sum() / len(ph))

def context_experience(name, hist, *, best_of=None, minutes_min=None, window=24, scale=8):
    ph = filter_player_history(name, hist, best_of=best_of, minutes_min=minutes_min)
    return float(min(len(ph.tail(window)) / scale, 1.0))

def context_avg(name, hist, *, column, best_of=None, minutes_min=None, window=12, min_matches=3, fallback=0.0):
    ph = filter_player_history(name, hist, best_of=best_of, minutes_min=minutes_min).tail(window)
    if len(ph) < min_matches:
        return fallback
    vals = pd.to_numeric(ph[column], errors="coerce").dropna()
    return float(vals.mean()) if len(vals) >= min_matches else fallback

## 4. Serve score — bogata charakterystyka serwisu

`compose_serve_score` agreguje 8 metryk serwisowych do jednego skalara w [0,1]:
- ace_rate / df_rate (asy vs double faults)
- first_in_pct / first_won_pct (skuteczność pierwszego serwisu)
- second_won_pct (drugi serwis pod presją)
- bp_save_pct (obrona break pointów)
- bp_faced_per_game (presja na serwisie)
- return_pts_won (jakość returnu)

Wagi dobrane tak, by faworyzować skuteczność punktów wygranych na serwisie i obronę break pointów.

**Stability** = 1/(1+std(serve_scores)) — wysoka stabilność oznacza powtarzalny serwis.

In [ ]:
def safe_ratio(num, denom, default=0.0):
    return float(num / denom) if denom > 0 else default

def extract_serve_metrics(match, player_name):
    is_w = match["winner_name"] == player_name
    pfx = "w" if is_w else "l"
    opp = "l" if is_w else "w"
    svpt = float(match[f"{pfx}_svpt"])
    first_in = float(match[f"{pfx}_1stIn"])
    second_serve = max(svpt - first_in, 0.0)
    return {
        "ace_rate": safe_ratio(float(match[f"{pfx}_ace"]), svpt, 0.08),
        "df_rate": safe_ratio(float(match[f"{pfx}_df"]), svpt, 0.03),
        "first_in_pct": safe_ratio(first_in, svpt, 0.60),
        "first_won_pct": safe_ratio(float(match[f"{pfx}_1stWon"]), first_in, 0.70),
        "second_won_pct": safe_ratio(float(match[f"{pfx}_2ndWon"]), second_serve, 0.50),
        "bp_save_pct": safe_ratio(float(match[f"{pfx}_bpSaved"]), float(match[f"{pfx}_bpFaced"]), 0.60),
        "bp_faced_per_game": safe_ratio(float(match[f"{pfx}_bpFaced"]), float(match[f"{pfx}_SvGms"]), 0.40),
        "return_pts_won": safe_ratio(
            float(match[f"{opp}_svpt"]) - float(match[f"{opp}_1stWon"]) - float(match[f"{opp}_2ndWon"]),
            float(match[f"{opp}_svpt"]), 0.35,
        ),
    }

def compose_serve_score(s):
    ace_c = min(s["ace_rate"] / 0.15, 1.5)
    df_c = min(s["df_rate"] / 0.08, 1.5)
    bp_c = min(s["bp_faced_per_game"] / 0.80, 1.5)
    return float(
        0.10 * ace_c + 0.18 * s["first_in_pct"] + 0.24 * s["first_won_pct"] + 0.22 * s["second_won_pct"]
        + 0.14 * s["bp_save_pct"] + 0.12 * s["return_pts_won"] - 0.08 * df_c - 0.08 * bp_c
    )

def context_serve_profile(name, hist, *, best_of=None, surface=None, opponent_rank_max=None, window=10, min_matches=2, fallback=None):
    if fallback is None:
        fallback = {"serve_score": 0.55, "return_score": 0.35, "stability": 0.50}
    ph = filter_player_history(name, hist, best_of=best_of, surface=surface, opponent_rank_max=opponent_rank_max).tail(window)
    if len(ph) < min_matches:
        return fallback
    metrics = [extract_serve_metrics(m, name) for _, m in ph.iterrows()]
    scores = [compose_serve_score(m) for m in metrics]
    return {
        "serve_score": float(np.mean(scores)),
        "return_score": float(np.mean([m["return_pts_won"] for m in metrics])),
        "stability": float(1.0 / (1.0 + np.std(scores))),
    }

## 5. Endurance score — agregat wytrzymałości

Najważniejsza nowość: jedna kompozytowa cecha mówiąca 'jak dobrze gracz radzi sobie z dystansem Bo5'. Łączy:
- formę w Bo5 (25%)
- formę w długich meczach >150 min (20%)
- doświadczenie Bo5 (15%)
- doświadczenie w długich meczach (10%)
- avg czas trwania Bo5 znormalizowany (15%)
- stabilność serwisu w Bo5 (15%)

In [ ]:
def endurance_score(*, bo5_form, long_form, bo5_exp, long_exp, bo5_minutes, bo5_stability):
    minutes_c = float(np.clip((bo5_minutes - 90.0) / 120.0, 0.0, 1.0))
    return float(
        0.25 * bo5_form + 0.20 * long_form + 0.15 * bo5_exp + 0.10 * long_exp
        + 0.15 * minutes_c + 0.15 * bo5_stability
    )

def build_fallback_serve(row, prefix):
    s = {k: float(row[f"{prefix}_{k}"]) for k in [
        "ace_rate", "df_rate", "first_in_pct", "first_won_pct",
        "second_won_pct", "bp_save_pct", "bp_faced_per_game", "return_pts_won",
    ]}
    return {"serve_score": compose_serve_score(s), "return_score": s["return_pts_won"], "stability": 0.50}

## 6. Główna pętla — liczenie cech kontekstowych

Dla każdego meczu:
1. Ustaw kontekst (index + cutoff) raz
2. Wywołaj ~12 funkcji kontekstowych dla winner i loser
3. Złóż endurance_score
4. Zapisz wszystko jako `w_*`/`l_*`

Po pętli wyczyść kontekst (`set_history_context(None, None)`).

In [ ]:
def add_targeted_slice_features(df_subset, historical_data, context_base_cols):
    df_subset = df_subset.copy()
    full_seq = pd.concat([historical_data[context_base_cols], df_subset[context_base_cols]], ignore_index=True)
    start_idx = len(historical_data)
    history_index = PlayerHistoryIndex(full_seq)

    rows = []
    for i in range(len(df_subset)):
        row = df_subset.iloc[i]
        cutoff = start_idx + i
        set_history_context(history_index, cutoff)
        past = full_seq.iloc[:cutoff]
        w_name, l_name, surface = row["winner_name"], row["loser_name"], row["surface"]
        w_form, l_form = float(row["w_form"]), float(row["l_form"])
        w_sf, l_sf = float(row["w_surface_form"]), float(row["l_surface_form"])

        w_general_min = context_avg(w_name, past, column="minutes", window=12, min_matches=3, fallback=110.0)
        l_general_min = context_avg(l_name, past, column="minutes", window=12, min_matches=3, fallback=110.0)
        w_bo5_form = context_form(w_name, past, best_of=5, window=8, min_matches=2, fallback=w_form)
        l_bo5_form = context_form(l_name, past, best_of=5, window=8, min_matches=2, fallback=l_form)
        w_bo5_exp = context_experience(w_name, past, best_of=5, window=20, scale=6)
        l_bo5_exp = context_experience(l_name, past, best_of=5, window=20, scale=6)
        w_bo5_min = context_avg(w_name, past, column="minutes", best_of=5, window=8, min_matches=2, fallback=w_general_min)
        l_bo5_min = context_avg(l_name, past, column="minutes", best_of=5, window=8, min_matches=2, fallback=l_general_min)
        w_long_form = context_form(w_name, past, minutes_min=150.0, window=10, min_matches=2, fallback=w_form)
        l_long_form = context_form(l_name, past, minutes_min=150.0, window=10, min_matches=2, fallback=l_form)
        w_long_exp = context_experience(w_name, past, minutes_min=150.0, window=20, scale=6)
        l_long_exp = context_experience(l_name, past, minutes_min=150.0, window=20, scale=6)
        w_fb_serve = build_fallback_serve(row, "w")
        l_fb_serve = build_fallback_serve(row, "l")
        w_bo5_serve = context_serve_profile(w_name, past, best_of=5, window=8, min_matches=2, fallback=w_fb_serve)
        l_bo5_serve = context_serve_profile(l_name, past, best_of=5, window=8, min_matches=2, fallback=l_fb_serve)
        w_pressure = w_bo5_serve if int(row["best_of"]) == 5 else w_fb_serve
        l_pressure = l_bo5_serve if int(row["best_of"]) == 5 else l_fb_serve

        rows.append({
            "w_best_of5_form": w_bo5_form, "l_best_of5_form": l_bo5_form,
            "w_best_of5_experience": w_bo5_exp, "l_best_of5_experience": l_bo5_exp,
            "w_best_of5_avg_minutes": w_bo5_min, "l_best_of5_avg_minutes": l_bo5_min,
            "w_long_match_form": w_long_form, "l_long_match_form": l_long_form,
            "w_long_match_experience": w_long_exp, "l_long_match_experience": l_long_exp,
            "w_best_of5_serve_score": w_bo5_serve["serve_score"], "l_best_of5_serve_score": l_bo5_serve["serve_score"],
            "w_best_of5_return_score": w_bo5_serve["return_score"], "l_best_of5_return_score": l_bo5_serve["return_score"],
            "w_best_of5_serve_stability": w_bo5_serve["stability"], "l_best_of5_serve_stability": l_bo5_serve["stability"],
            "w_pressure_serve_score": w_pressure["serve_score"], "l_pressure_serve_score": l_pressure["serve_score"],
            "w_endurance_score": endurance_score(bo5_form=w_bo5_form, long_form=w_long_form, bo5_exp=w_bo5_exp,
                                                  long_exp=w_long_exp, bo5_minutes=w_bo5_min, bo5_stability=w_bo5_serve["stability"]),
            "l_endurance_score": endurance_score(bo5_form=l_bo5_form, long_form=l_long_form, bo5_exp=l_bo5_exp,
                                                  long_exp=l_long_exp, bo5_minutes=l_bo5_min, bo5_stability=l_bo5_serve["stability"]),
            "tourney_level_raw": row["tourney_level"],
        })

    set_history_context(None, None)
    return pd.concat([df_subset.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

df_train_raw = add_targeted_slice_features(df_train_raw, history_ctx, context_base_cols)
hist_v = pd.concat([history_ctx, df_train_raw[context_base_cols]], ignore_index=True)
df_val_raw = add_targeted_slice_features(df_val_raw, hist_v, context_base_cols)
hist_t = pd.concat([history_ctx, df_train_raw[context_base_cols], df_val_raw[context_base_cols]], ignore_index=True)
df_test_raw = add_targeted_slice_features(df_test_raw, hist_t, context_base_cols)

print("Cechy kontekstowe Bo5 + endurance policzone dla train/val/test")

## 7. Symetryzacja + p1/p2 mapping

Identyczna logika jak w SliceAware — `np.where(y==1, w_*, l_*)`. Dodatkowo `is_best_of5` jako binarna flaga i `tourney_level_strength` jako numeryczny.

In [ ]:
SYMMETRIC_FEATURE_SPECS = [
    ("best_of5_form", "best_of5_form_diff"),
    ("best_of5_experience", "best_of5_experience_diff"),
    ("best_of5_avg_minutes", "best_of5_avg_minutes_diff"),
    ("long_match_form", "long_match_form_diff"),
    ("long_match_experience", "long_match_experience_diff"),
    ("best_of5_serve_score", "best_of5_serve_score_diff"),
    ("best_of5_return_score", "best_of5_return_score_diff"),
    ("best_of5_serve_stability", "best_of5_serve_stability_diff"),
    ("pressure_serve_score", "pressure_serve_score_diff"),
    ("endurance_score", "endurance_score_diff"),
]

def attach_targeted_features(sym, raw):
    helper = ["match_id", "tourney_level_raw"]
    for name, _ in SYMMETRIC_FEATURE_SPECS:
        helper.extend([f"w_{name}", f"l_{name}"])
    e = sym.merge(raw[helper], on="match_id", how="left", validate="many_to_one")
    mask = e["y"] == 1
    e["is_best_of5"] = (e["best_of"] == 5).astype(int)
    e["tourney_level_strength"] = e["tourney_level_raw"].map(TOURNEY_LEVEL_STRENGTH).fillna(0.60).astype(float)
    e["best_of5_level_pressure"] = e["is_best_of5"] * e["tourney_level_strength"]
    for name, diff in SYMMETRIC_FEATURE_SPECS:
        p1, p2 = f"p1_{name}", f"p2_{name}"
        e[p1] = np.where(mask, e[f"w_{name}"], e[f"l_{name}"])
        e[p2] = np.where(mask, e[f"l_{name}"], e[f"w_{name}"])
        e[diff] = e[p1] - e[p2]
    drop = ["tourney_level_raw"]
    for name, _ in SYMMETRIC_FEATURE_SPECS:
        drop.extend([f"w_{name}", f"l_{name}"])
    return e.drop(columns=drop)

train_data = attach_targeted_features(symmetrize_data(df_train_raw, shuffle=True), df_train_raw)
val_data = attach_targeted_features(symmetrize_data(df_val_raw, shuffle=True), df_val_raw)
test_data = attach_targeted_features(symmetrize_data(df_test_raw, shuffle=True), df_test_raw)

TARGETED = ["is_best_of5", "tourney_level_strength", "best_of5_level_pressure"]
for name, diff in SYMMETRIC_FEATURE_SPECS:
    TARGETED.extend([f"p1_{name}", f"p2_{name}", diff])

features = list(namespace["features"]) + TARGETED
print(f"Liczba cech: {len(features)} (baseline: {len(namespace['features'])}, dodane: {len(TARGETED)})")

## 8. Trening + porównanie z baseline

In [ ]:
X_train, y_train = train_data[features], train_data["y"]
X_val, y_val = val_data[features], val_data["y"]
X_test, y_test = test_data[features], test_data["y"]

best_rf = RandomForestClassifier(**baseline_search.best_params_, n_jobs=-1, random_state=namespace["RANDOM_STATE"])
best_rf.fit(X_train, y_train)

val_acc = accuracy_score(y_val, best_rf.predict(X_val))
test_proba = best_rf.predict_proba(X_test)
test_acc = accuracy_score(y_test, best_rf.predict(X_test))

test_data["p1_win_probability"] = test_proba[:, 1]
wp = test_data[test_data["y"] == 1]
match_accuracy = float((wp["p1_win_probability"] > 0.5).mean())

print("=== POROWNANIE Z BASELINE ===")
print(f"Validation:  baseline={baseline_val_acc:.4f} | bestof5_v1={val_acc:.4f} | delta={val_acc - baseline_val_acc:+.4f}")
print(f"Test:        baseline={baseline_test_acc:.4f} | bestof5_v1={test_acc:.4f} | delta={test_acc - baseline_test_acc:+.4f}")
print(f"Match-level: baseline={baseline_match_accuracy:.4f} | bestof5_v1={match_accuracy:.4f} | delta={match_accuracy - baseline_match_accuracy:+.4f}")

## 9. Ważność nowych cech

Kluczowe pytanie: czy `endurance_score` faktycznie znalazł się w top cech? Jeśli tak, to znaczy że nasza intuicja o specyfice Bo5 była trafna.

In [ ]:
fi = pd.DataFrame({"feature": features, "importance": best_rf.feature_importances_})
new_fi = fi[fi["feature"].isin(TARGETED)].sort_values("importance", ascending=False)
print("=== WAZNOSC NOWYCH CECH (top 15) ===")
print(new_fi.head(15).to_string(index=False))

## 10. Wnioski

**Realny wynik (RANDOM_STATE=42)**:
- Baseline match accuracy: **61.02%**
- BestOf5 v1 match accuracy: **63.39%**
- **Delta: +2.37 p.p.** -- NAJLEPSZY wynik spośród wszystkich wariantów slice-aware

**Co dało Bo5-specific feature engineering:**

Największe zyski accuracy na konkretnych slice'ach:
- `rank_gap=0-10 x age_gap=>8` (top vs top, duża różnica wieku): **+33.3 p.p.**
- `round=R128 x L-vs-R` (pierwsza runda Grand Slamu z lewo): **+33.3 p.p.**
- `tourney_level=D x rank_gap=0-10` (Davis Cup, wyrównane): +23.1 p.p.
- `tourney_level=M x rank_gap=0-10` (Masters, wyrównane): +21.7 p.p.
- `L-vs-R x rank_gap=0-10`: +21.4 p.p.
- `rank_gap=0-10 x age_gap=3-5`: +20.0 p.p.

Cechy `endurance_score_diff`, `best_of5_serve_score_diff`, `tourney_level_strength` wchodzą do top 30 ważności -- model faktycznie rozpoznaje je jako użyteczne.

**Największe spadki accuracy** (gdzie wariant zaszkodził):
- `tourney_level=F x age_gap=0-2`: -50.0 p.p. (ale support tylko 6)
- `round=F x age_gap=0-2`: -40.0 p.p. (support 5)
- `round=R128 x rank_gap=26-50`: -33.3 p.p.
- `tourney_level=F`: -20.0 p.p.

Te spadki są skupione na małych slice'ach (support 5-15), więc mogą być szumem. Wilson CI na test secie pokazałby, czy to istotne czy nie.

**Ograniczenia:**
- Bo5 stanowi tylko ~18% danych (Grand Slamy + Davis Cup). Wariant najbardziej pomaga tym meczom, ale globalne accuracy zysk +2.37 p.p. odzwierciedla głównie tamtą poprawę rozcieńczoną na cały test set.
- `endurance_score` jest własną kompozycją wag (25/20/15/10/15/15) -- wagi dobrane intuicyjnie, nie przez optymalizację. Możliwe że inne kombinacje są lepsze.
- Brak `tourney_id` w tym wariancie -- nie wykorzystujemy informacji o tym, ile meczów gracz już rozegrał w danym turnieju (drabince).

**Następny krok:** `qfserve_v3` dodaje właśnie `tourney_id`/seed/draw_size + warunkowe statystyki serwisu. Realny wynik tamtego notebooka: +2.20 p.p. -- minimalnie gorszy od BestOf5 v1, ale przełamuje slice'y na inne sposoby (np. `round=R128 x L-vs-R` +44.4 p.p. tam vs +33.3 p.p. tutaj).